# Workflow spatialmeta horizontal integration

## Packages

In [ ]:
import spatialmeta as smt
from spatialmeta.model import ConditionalVAESM
import numpy as np
import pandas as pd
import scanpy as sc
import anndata
import torch
import copy
from pyimzml.ImzMLParser import ImzMLParser

import matplotlib.pyplot as plt
import seaborn as sns

import warnings
warnings.filterwarnings("ignore")

## Download data

In [ ]:
RCC_sample_list = ["R114_T", "S15_T", "X49_T", "Y27_T", "Y7_T"]

#Test data can be obtained from SpatialMETA
for i in RCC_sample_list:
    adata_SM = smt.data.load_adata(
        sample_name= i+"_raw",
        modality="SM"
    )
    adata_name = "adata_SM_"+i
    globals()[adata_name] = adata_SM

In [ ]:
merge_SM_adata = anndata.concat({
"R114_T":adata_SM_R114_T,
"S15_T":adata_SM_S15_T,
"X49_T":adata_SM_X49_T,
"Y27_T":adata_SM_Y27_T,
"Y7_T":adata_SM_Y7_T,
},label="sample")


## Exploration

### Quality metrics

In [ ]:
for i in RCC_sample_list:
    print(f"\nSM dataset: {i}")
    smt.pp.calculate_qc_metrics_sm(globals()["adata_SM_"+i])
    print(globals()["adata_SM_"+i])
    
print(f"\nSM merged dataset:")
merge_SM_adata


### Total intensity vs mean intensity

In [ ]:

for i in RCC_sample_list:
    print(f"\nSM dataset: {i}")
    adata_SM = globals()["adata_SM_"+i]
    
    sc.pl.violin(
        adata_SM,
        ['total_intensity', 'mean_intensity'],
        jitter=0.4,
        multi_panel=True,
    )


print(f"\nSM dataset: merged")
sc.pl.violin(
        merge_SM_adata,
        ['total_intensity', 'mean_intensity'],
        jitter=0.4,
        multi_panel=True,
    )


### Embedding

In [ ]:
from matplotlib.colors import LinearSegmentedColormap

def make_colormap(colors):
    return LinearSegmentedColormap.from_list("custom", colors)

for i in RCC_sample_list:
    print(f"\nSM dataset: {i}")
    adata_SM = globals()["adata_SM_"+i]
    sc.pl.embedding(adata_SM,
                    basis='spatial',
                    cmap = make_colormap(['#3c096c','#FFFFFF','#ff6b35']),
                    color=['total_intensity','mean_intensity'])
    

## Pre-processing

### Filter spots with low intensity

In [ ]:
intensity_threshold = {"R114_T": 1e6,
                        "S15_T": 1e6,
                        "X49_T": 1e6,
                        "Y27_T": 1e6,
                        "Y7_T": 2e6}


for i in intensity_threshold:
    print(f"\nFiltering SM dataset: {i}")
    adata_SM = globals()["adata_SM_"+i]
    adata_SM = smt.pp.filter_cells_sm(adata_SM,min_total_intensity=intensity_threshold[i])

    globals()["adata_SM_"+i] = adata_SM

### Merge and normalize merged data

In [ ]:
merge_SM_adata = anndata.concat({
"R114_T":adata_SM_R114_T,
"S15_T":adata_SM_S15_T,
"X49_T":adata_SM_X49_T,
"Y27_T":adata_SM_Y27_T,
"Y7_T":adata_SM_Y7_T,
},label="sample")

In [ ]:
merge_SM_adata.layers["counts"] = merge_SM_adata.X.copy()

sc.pp.normalize_total(
    merge_SM_adata,
    target_sum = 1e4
)

merge_SM_adata.layers["normalized"] = merge_SM_adata.X.copy()

### Calculate and remove batch-biased features

In [ ]:
smt.pp.spatial_variable(merge_SM_adata,
                        n_top_variable=800,
                        add_key = "highly_variable_moranI",
                        batch_key="sample",
                        min_samples = 3,
                        min_frac = 0.9,
                        min_logfc= 3
                       )

merge_SM_adata = merge_SM_adata[:,merge_SM_adata.var.highly_variable_moranI]

merge_SM_adata

## Horizontal integration

In [ ]:
merge_SM_adata.X = merge_SM_adata.layers['normalized']
merge_SM_adata.var['type'] = "SM"

model = smt.model.ConditionalVAESM(merge_SM_adata,
                                       n_latent=10,
                                       device='cpu',
                                       batch_keys="sample",
                                       batch_embedding="embedding"
                                      )


loss_dict = model.fit(
    max_epoch=64,
    lr=1e-3,
    kl_loss_reduction= 'mean',
    kl_weight = 15,
    n_epochs_kl_warmup = 0
)

## Visualization and analysis

In [ ]:
Z = model.get_latent_embedding()
X = model.get_normalized_expression()

merge_SM_adata.layers['reconstruction'] = X
merge_SM_adata.obsm['X_emb'] = np.vstack(Z)

In [ ]:
sc.pp.neighbors(merge_SM_adata,use_rep="X_emb",n_neighbors=15)
sc.tl.umap(merge_SM_adata,
          min_dist=1,spread=1)
sc.tl.leiden(merge_SM_adata,
             key_added="VAE_clusters_latent10")

In [ ]:
sc.settings.set_figure_params(dpi=200, facecolor="white")
fig,ax=plt.subplots()
fig.set_size_inches(5,5)
sc.pl.umap(merge_SM_adata,color=["sample"],
          palette = {
              "R114_T": "#F97300",
              "S15_T": "#7469B6",
              "X49_T": "#7ABA78",
              "Y27_T": "#03AED2",
              "Y7_T": "#F3CA52",
          },
          show = True,
          size = 3,
          ax=ax)

In [ ]:
print(merge_SM_adata.obs.columns.tolist())

In [ ]:
sc.settings.set_figure_params(dpi=200, facecolor="white")
fig,ax=plt.subplots()
fig.set_size_inches(5,5)
sc.pl.umap(merge_SM_adata,color=["VAE_clusters_latent10"],
          palette = sc.pl.palettes.godsnot_102[40:],
          show = True,
          size = 3,
          ax=ax)